# Extension 04 Differentiable goal optimisation

In [ ]:
%pip install -q numpy session-py==0.310.0 session-jupyter==0.2.0

# Question

**How can we differentiate the Force Density Method with respect to $q$ to match target shapes, equalise cable forces, and planarise quad gridshell panels?**

Inspired by the working solvers in `arpastrana/jax_fdm` (`equilibrium/models.py`, `goals/`, `examples/truss/truss_equal_force.py`, `examples/gridshell/gridshell_planarization.py`), `arpastrana/dfdm`, `PrincetonLIPS/neural_fdm`, `efradeca/vae-fdm` (`classical.py`), and `jikempff/Mite` (`Planarization.cs`).

```
n42 ─ n43 ─ n44 ─ n45 ─ n46 ─ n47 ─ n48
 │     │     │     │     │     │     │
n35 ─ n36 ─ n37 ─ n38 ─ n39 ─ n40 ─ n41
 │     │     │     │     │     │     │
n28 ─ n29 ─ n30 ─ n31 ─ n32 ─ n33 ─ n34
 │     │     │     │     │     │     │
n21 ─ n22 ─ n23 ─ n24 ─ n25 ─ n26 ─ n27
 │     │     │     │     │     │     │
n14 ─ n15 ─ n16 ─ n17 ─ n18 ─ n19 ─ n20
 │     │     │     │     │     │     │
n7  ─ n8  ─ n9  ─ n10 ─ n11 ─ n12 ─ n13
 │     │     │     │     │     │     │
n0  ─ n1  ─ n2  ─ n3  ─ n4  ─ n5  ─ n6
```

**Given:** differentiating the equilibrium equation $D_N\,{\color{#d62828} x_N} + D_F\,{\color{#2a6fdb} x_F} = {\color{#2e9e4a} p_x}$ (where $D_N = C_N^{T}\,Q\,C_N$, $D_F = C_N^{T}\,Q\,C_F$, and $u = C_N\,{\color{#d62828} x_N} + C_F\,{\color{#2a6fdb} x_F}$) with respect to $q$ yields the exact analytical sensitivity Jacobian:

$$
D_N\,\frac{\partial {\color{#d62828} x_N}}{\partial q} + C_N^{T}\,U = 0 \quad \Longrightarrow \quad J_x = \frac{\partial {\color{#d62828} x_N}}{\partial q} = -D_N^{-1}\,C_N^{T}\,U
$$

with $J_y = -D_N^{-1}\,C_N^{T}\,V$ and $J_z = -D_N^{-1}\,C_N^{T}\,W$. Chain differentiation gives the exact Jacobians of bar lengths $l$ and bar forces $f = Q\,l$:

$$
J_l = \frac{\partial l}{\partial q} = L^{-1}\left(U\,C_N\,J_x + V\,C_N\,J_y + W\,C_N\,J_z\right), \qquad J_f = \frac{\partial f}{\partial q} = L + Q\,J_l
$$

<div align="center">
  <img src="https://raw.githubusercontent.com/BRG-teaching/Teaching/main/CAS/logos/fd_coordinate_difference.svg"
       style="width:100%; height:auto;"
       alt="Coordinate difference u = C x along a bar linking force density variations to node displacements" />
  <p style="text-align:center;">
    <em>Coordinate differences u, v, w assemble the diagonal matrices U, V, W that drive the analytical sensitivity Jacobian.</em>
  </p>
</div>

- Derive and verify $J_x, J_y, J_z, J_l, J_f$ against central finite differences to $10^{-9}$
- Match a target dome elevation while equalising interior bar lengths with edge group parameter tying (`VertexPointGoal` and `EdgeLengthGoal` in `jax_fdm`)
- Equalise every bottom cable force in a cable stayed truss while keeping the top deck level (`truss_equal_force.py`)
- Planarise all quad faces of a compression gridshell for flat glass cladding (`gridshell_planarization.py` and `Mite`)

<hr style="border: none; border-top: 2px solid #555; margin: 2em 0;">

# Analytical Sensitivity Jacobians

We write a single reusable function `fdm_sensitivities` that solves `# equation 6.32` for any network and returns $(x, y, z, l, f)$ together with the exact analytical Jacobians $(J_x, J_y, J_z, J_l, J_f)$ using $U = \operatorname{diag}(u)$, $V = \operatorname{diag}(v)$, $W = \operatorname{diag}(w)$, and $L = \operatorname{diag}(l)$. Let us check its accuracy against central finite differences on the $6 \times 6$ grid.

In [1]:
import numpy as np
from session_py import Color, Matrix, Session
from session_jupyter import show

RED = Color(0.84, 0.16, 0.16, 1.0)
BLUE = Color(0.16, 0.44, 0.86, 1.0)
GREEN = Color(0.18, 0.62, 0.29, 1.0)
GRAY = Color(0.74, 0.74, 0.74, 1.0)


def fdm_sensitivities(C_mat, free_idx, fixed_idx, xyz_init, p_nodes, q_vec):
    """Solve FDM equilibrium and compute exact analytical Jacobians d(x_N, y_N, z_N, l, f)/dq."""
    C_N = C_mat[:, free_idx]
    C_F = C_mat[:, fixed_idx]
    Q = np.diag(q_vec)
    x_new, y_new, z_new = xyz_init[:, 0].copy(), xyz_init[:, 1].copy(), xyz_init[:, 2].copy()

    # equation 6.32
    A = C_N.T @ Q @ C_N
    x_new[free_idx] = np.linalg.solve(A, p_nodes[free_idx, 0] - C_N.T @ Q @ C_F @ x_new[fixed_idx])
    y_new[free_idx] = np.linalg.solve(A, p_nodes[free_idx, 1] - C_N.T @ Q @ C_F @ y_new[fixed_idx])
    z_new[free_idx] = np.linalg.solve(A, p_nodes[free_idx, 2] - C_N.T @ Q @ C_F @ z_new[fixed_idx])

    u = C_mat @ x_new
    v = C_mat @ y_new
    w = C_mat @ z_new
    l = np.sqrt(u**2 + v**2 + w**2)
    f = q_vec * l

    # analytical sensitivity Jacobians: D_N J_x = -C_N^T U
    U, V, W = np.diag(u), np.diag(v), np.diag(w)
    J_x = -np.linalg.solve(A, C_N.T @ U)
    J_y = -np.linalg.solve(A, C_N.T @ V)
    J_z = -np.linalg.solve(A, C_N.T @ W)

    J_l = (U @ C_N @ J_x + V @ C_N @ J_y + W @ C_N @ J_z) / l[:, None]
    J_f = np.diag(l) + Q @ J_l

    xyz_out = np.column_stack([x_new, y_new, z_new])
    return xyz_out, l, f, J_x, J_y, J_z, J_l, J_f


# build 6 x 6 grid (49 nodes, 84 bars) and verify Jacobians against central finite differences
n = 7
gx, gy = np.meshgrid(range(n), range(n))
x0, y0 = gx.flatten().astype(float), gy.flatten().astype(float)
xyz0 = np.column_stack([x0, y0, np.zeros(n * n)])

edges = [(k, k + 1) for k in range(n * n) if x0[k] < n - 1] + [
    (k, k + n) for k in range(n * n) if y0[k] < n - 1
]
C = np.zeros((len(edges), n * n))
for e, (a, b) in enumerate(edges):
    C[e, a], C[e, b] = 1.0, -1.0

corners = [0, n - 1, n * (n - 1), n * n - 1]
free = [k for k in range(n * n) if k not in corners]
p_grid = np.zeros((n * n, 3))
p_grid[free, 2] = -0.25

rng = np.random.default_rng(7)
q_test = rng.uniform(1.0, 3.0, size=len(edges))
_, _, _, Jx_a, _, Jz_a, Jl_a, Jf_a = fdm_sensitivities(C, free, corners, xyz0, p_grid, q_test)

eps = 1e-6
Jz_fd = np.zeros_like(Jz_a)
Jf_fd = np.zeros_like(Jf_a)
for k in range(len(edges)):
    dq = np.zeros_like(q_test)
    dq[k] = eps
    xyz_p, _, f_p, *_ = fdm_sensitivities(C, free, corners, xyz0, p_grid, q_test + dq)
    xyz_m, _, f_m, *_ = fdm_sensitivities(C, free, corners, xyz0, p_grid, q_test - dq)
    Jz_fd[:, k] = (xyz_p[free, 2] - xyz_m[free, 2]) / (2.0 * eps)
    Jf_fd[:, k] = (f_p - f_m) / (2.0 * eps)

print("max |Jz_analytical - Jz_fd| =", f"{np.abs(Jz_a - Jz_fd).max():.2e}")
print("max |Jf_analytical - Jf_fd| =", f"{np.abs(Jf_a - Jf_fd).max():.2e}")

max |Jz_analytical - Jz_fd| = 1.27e-09
max |Jf_analytical - Jf_fd| = 5.97e-09


# Target Crown Shape and Length Equalisation with Edge Group Tying

In `jax_fdm` (`parameters/` and `goals/`) and `vae-fdm` (`classical.py`), design variables $\theta$ are often tied across symmetric cable groups $q = G\,\theta$ inside sign preserving bounds $\theta \in [q_{\min}, q_{\max}]$. Using Gauss Newton steps with our exact Jacobian $J_r = \frac{\partial r}{\partial q}\,G$, we optimise symmetric cable rings on our $6 \times 6$ tension net so the central 9 nodes match a target sag profile $z_{\text{target}} = -1.35\text{ m}$ (`VertexPointGoal`) while interior bar lengths equalise toward $0.95\text{ m}$ (`EdgeLengthGoal`).

In [2]:
# group each bar by sorted distance of its midpoint from the grid centre (3, 3) -> symmetric parameter tying q = G theta
group_keys = []
for a, b in edges:
    mx = abs(0.5 * (x0[a] + x0[b]) - 3.0)
    my = abs(0.5 * (y0[a] + y0[b]) - 3.0)
    group_keys.append((round(min(mx, my), 3), round(max(mx, my), 3)))
unique_groups = sorted(set(group_keys))
G = np.zeros((len(edges), len(unique_groups)))
for e, gk in enumerate(group_keys):
    G[e, unique_groups.index(gk)] = 1.0

is_interior_edge = np.array([
    not ((x0[a] in (0.0, 6.0) and x0[b] == x0[a]) or (y0[a] in (0.0, 6.0) and y0[b] == y0[a]))
    for a, b in edges
])
centre_patch = [16, 17, 18, 23, 24, 25, 30, 31, 32]
centre_free_pos = [free.index(k) for k in centre_patch]

theta = np.full(len(unique_groups), 1.5)
xyz_init_net, l_init_net, *_ = fdm_sensitivities(C, free, corners, xyz0, p_grid, G @ theta)

for step in range(40):
    q_cur = G @ theta
    xyz_c, l_c, f_c, Jx, Jy, Jz, Jl, _ = fdm_sensitivities(C, free, corners, xyz0, p_grid, q_cur)

    # residual vector combining VertexPointGoal (weight 4.0) and interior EdgeLengthGoal (weight 1.0)
    r_z = 4.0 * (xyz_c[centre_patch, 2] - (-1.35))
    r_l = 1.0 * (l_c[is_interior_edge] - 0.95)
    res_vec = np.concatenate([r_z, r_l])

    Jr_q = np.vstack([4.0 * Jz[centre_free_pos, :], 1.0 * Jl[is_interior_edge, :]])
    Jr_theta = Jr_q @ G

    # damped Gauss Newton step + box projection [0.25, 25.0] kN/m
    d_theta = -np.linalg.solve(Jr_theta.T @ Jr_theta + 1e-3 * np.eye(len(theta)), Jr_theta.T @ res_vec)
    theta = np.clip(theta + 0.8 * d_theta, 0.25, 25.0)

xyz_opt1, l_opt1, f_opt1, *_ = fdm_sensitivities(C, free, corners, xyz0, p_grid, G @ theta)
print("number of tied cable groups =", len(unique_groups), " (down from 84 bars)")
print("centre z24: initial =", round(xyz_init_net[24, 2], 3), "m -> optimised =", round(xyz_opt1[24, 2], 3), "m (target -1.350 m)")
print("interior length std: initial =", round(l_init_net[is_interior_edge].std(), 4), "m -> optimised =", round(l_opt1[is_interior_edge].std(), 4), "m")

colors_1 = [Color(0.65 - 0.65 * fv / f_opt1.max(), 0.65 - 0.65 * fv / f_opt1.max(), 0.65 - 0.65 * fv / f_opt1.max(), 1.0) for fv in f_opt1]
session_1 = Session("fdm_ext04_goal_net")
session_1.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(xyz_init_net.tolist()), [GRAY] * len(edges), [1.4] * len(edges))
session_1.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(xyz_opt1.tolist()), colors_1, [2.8] * len(edges))
for net in range(2):
    for k in range(n * n):
        pt = session_1.objects.points[net * n * n + k]
        pt.pointcolor = BLUE if k in corners else RED
        pt.width = 13.0 if k in corners else 6.5

show(session_1, "fdm_ext04_goal_net")

number of tied cable groups = 12  (down from 84 bars)
centre z24: initial = -1.748 m -> optimised = -1.398 m (target -1.350 m)
interior length std: initial = 0.1455 m -> optimised = 0.0143 m


# Equal Force Bottom Chord Truss

Now we reproduce the classic `jax_fdm` example `examples/truss/truss_equal_force.py` in pure NumPy! A $10\text{ m}$ span lenticular / bowstring truss has $9$ free top deck nodes carrying ${\color{#2e9e4a} p_z = -0.4\text{ kN}}$ each, a top compression chord ($q \lt 0$), vertical compression struts ($q \lt 0$), and a $10$ segment bottom tension cable ($q \gt 0$). With uniform initial $q$, the bottom cable force varies along the span and the top deck sags. Using $J_f$ (`EdgeForceGoal`) and $J_z$ (`NodesColinearGoal`), we drive **every single bottom chord segment** to the exact same tension $f_{\text{target}} = 3.20\text{ kN}$ while keeping the top deck level at $z = 0\text{ m}$!

In [3]:
num_seg = 10
xs_truss = np.linspace(-5.0, 5.0, num_seg + 1)
# nodes 0..10 = top chord (0 and 10 pinned at z = 0), nodes 11..19 = 9 bottom cable nodes
xyz_tr0 = np.zeros((2 * num_seg, 3))
xyz_tr0[: num_seg + 1, 0] = xs_truss
xyz_tr0[num_seg + 1 :, 0] = xs_truss[1:-1]
xyz_tr0[num_seg + 1 :, 2] = -1.5

top_edges = [(i, i + 1) for i in range(num_seg)]
bot_chain = [0] + list(range(num_seg + 1, 2 * num_seg)) + [num_seg]
bot_edges = [(bot_chain[i], bot_chain[i + 1]) for i in range(num_seg)]
strut_edges = [(num_seg + 1 + i, i + 1) for i in range(num_seg - 1)]
tr_edges = top_edges + bot_edges + strut_edges

C_tr = np.zeros((len(tr_edges), len(xyz_tr0)))
for e, (a, b) in enumerate(tr_edges):
    C_tr[e, a], C_tr[e, b] = 1.0, -1.0

fixed_tr = [0, num_seg]
free_tr = [k for k in range(len(xyz_tr0)) if k not in fixed_tr]
top_free_nodes = list(range(1, num_seg))
top_free_pos = [free_tr.index(k) for k in top_free_nodes]
bot_edge_idx = list(range(num_seg, 2 * num_seg))

p_tr = np.zeros((len(xyz_tr0), 3))
p_tr[top_free_nodes, 2] = -0.40

# initial force densities: top chord -2.5, bottom cable +3.0, vertical struts -0.6
q_tr0 = np.array([-2.5] * num_seg + [3.0] * num_seg + [-0.6] * (num_seg - 1))
q_low = np.array([-30.0] * num_seg + [0.2] * num_seg + [-15.0] * (num_seg - 1))
q_high = np.array([-0.1] * num_seg + [30.0] * num_seg + [-0.02] * (num_seg - 1))

xyz_tr_guess, _, f_tr_guess, *_ = fdm_sensitivities(C_tr, free_tr, fixed_tr, xyz_tr0, p_tr, q_tr0)

q_tr = q_tr0.copy()
target_f = 3.20
for step in range(60):
    xyz_tc, _, f_tc, _, _, Jz_t, _, Jf_t = fdm_sensitivities(C_tr, free_tr, fixed_tr, xyz_tr0, p_tr, q_tr)
    res_tr = np.concatenate([f_tc[bot_edge_idx] - target_f, 3.0 * xyz_tc[top_free_nodes, 2]])
    J_tr = np.vstack([Jf_t[bot_edge_idx, :], 3.0 * Jz_t[top_free_pos, :]])
    dq = -np.linalg.solve(J_tr.T @ J_tr + 1e-4 * np.eye(len(q_tr)), J_tr.T @ res_tr)
    q_tr = np.clip(q_tr + 0.7 * dq, q_low, q_high)
    if np.linalg.norm(res_tr) < 1e-8:
        break

xyz_tr_opt, _, f_tr_opt, *_ = fdm_sensitivities(C_tr, free_tr, fixed_tr, xyz_tr0, p_tr, q_tr)
print("initial bottom chord forces (kN)   : min =", round(f_tr_guess[bot_edge_idx].min(), 3), " max =", round(f_tr_guess[bot_edge_idx].max(), 3))
print("optimised bottom chord forces (kN) : min =", round(f_tr_opt[bot_edge_idx].min(), 5), " max =", round(f_tr_opt[bot_edge_idx].max(), 5))
print("max top deck vertical deviation    =", f"{np.abs(xyz_tr_opt[top_free_nodes, 2]).max():.2e}", "m")

colors_tr = [BLUE if fv < 0 else Color(0.15, 0.15, 0.15, 1.0) for fv in f_tr_opt]
widths_tr = [3.6 if fv < 0 else 3.0 for fv in f_tr_opt]

session_tr = Session("fdm_ext04_equal_force_truss")
session_tr.add_network(Matrix.from_rows(C_tr.tolist()), Matrix.from_rows(xyz_tr_guess.tolist()), [GRAY] * len(tr_edges), [1.5] * len(tr_edges))
session_tr.add_network(Matrix.from_rows(C_tr.tolist()), Matrix.from_rows(xyz_tr_opt.tolist()), colors_tr, widths_tr)
for net in range(2):
    for k in range(len(xyz_tr0)):
        pt = session_tr.objects.points[net * len(xyz_tr0) + k]
        pt.pointcolor = BLUE if k in fixed_tr else RED
        pt.width = 12.0 if k in fixed_tr else 7.0

show(session_tr, "fdm_ext04_equal_force_truss")

initial bottom chord forces (kN)   : min = 3.029  max = 4.12
optimised bottom chord forces (kN) : min = 3.2  max = 3.2
max top deck vertical deviation    = 1.92e-10 m


# Compression Gridshell Quad Planarisation

Finally we implement `jax_fdm/examples/gridshell/gridshell_planarization.py` (`MeshPlanarityGoal`) and `Mite` (`Planarization.cs`). A quad face $(v_0, v_1, v_2, v_3)$ is planar if and only if its two diagonals $d_1 = x_{v_2} - x_{v_0}$ and $d_2 = x_{v_3} - x_{v_1}$ intersect, meaning the signed skew volume $(x_{v_1} - x_{v_0}) \cdot (d_1 \times d_2)$ vanishes! Normalising the diagonal distance by $(d_1 \times d_2)\,\bar{l}$ gives the dimensionless flatness error:

$$
\delta_f(x) = \frac{\left(x_{v_1} - x_{v_0}\right) \cdot \left(d_1 \times d_2\right)}{\|d_1 \times d_2\|\,\bar{l}_f}
$$

When we pin the four corners plus one boundary side $x = 6\text{ m}$ of a compression gridshell ($q \lt 0$), the initial funicular shell has badly warped panels near the open corners. Differentiating $\delta_f(x(q))$ through $J_x, J_y, J_z$ drives **all 36 quad panels** well inside the $1\%$ flat glass tolerance while keeping every bar in pure compression!

In [4]:
faces_q = [
    (j * n + i, j * n + i + 1, (j + 1) * n + i + 1, (j + 1) * n + i)
    for j in range(n - 1)
    for i in range(n - 1)
]

# pin the 4 corners AND the full right side x = 6 m (matching gridshell_planarization.py)
supports_pl = sorted(set(corners + [k for k in range(n * n) if x0[k] == 6.0]))
free_pl = [k for k in range(n * n) if k not in supports_pl]
free_pl_map = {k: idx for idx, k in enumerate(free_pl)}

p_pl = np.zeros((n * n, 3))
p_pl[free_pl, 2] = -0.35

q_pl0 = np.full(len(edges), -1.0)
for e, (a, b) in enumerate(edges):
    on_bnd = (x0[a] in (0.0, 6.0) and x0[b] == x0[a]) or (y0[a] in (0.0, 6.0) and y0[b] == y0[a])
    both_sup = (a in supports_pl) and (b in supports_pl)
    if on_bnd and not both_sup:
        q_pl0[e] = -4.5


def quad_flatness_and_jac(xyz, Jx, Jy, Jz):
    """Signed normalized diagonal distance per quad face and its exact Jacobian d(delta)/dq."""
    m_q = Jx.shape[1]
    deltas = np.zeros(len(faces_q))
    J_delta = np.zeros((len(faces_q), m_q))

    def dnode(vid):
        if vid not in free_pl_map:
            return np.zeros((3, m_q))
        row = free_pl_map[vid]
        return np.vstack([Jx[row], Jy[row], Jz[row]])

    for fi, (i0, i1, i2, i3) in enumerate(faces_q):
        p0, p1, p2, p3 = xyz[i0], xyz[i1], xyz[i2], xyz[i3]
        d1 = p2 - p0
        d2 = p3 - p1
        cr = np.cross(d1, d2)
        cr_norm = np.linalg.norm(cr) + 1e-12
        n_hat = cr / cr_norm
        l_bar = 0.25 * (
            np.linalg.norm(p1 - p0) + np.linalg.norm(p2 - p1) + np.linalg.norm(p3 - p2) + np.linalg.norm(p0 - p3)
        )
        # signed distance between diagonals normalized by mean edge length
        deltas[fi] = np.dot(p1 - p0, n_hat) / l_bar

        # exact first order sensitivity treating unit normal n_hat and l_bar locally frozen (zero at delta = 0!)
        dp0, dp1, dp2, dp3 = dnode(i0), dnode(i1), dnode(i2), dnode(i3)
        dd1 = dp2 - dp0
        dd2 = dp3 - dp1
        # d((p1 - p0) . cr) = cr . (dp1 - dp0) + ((p1 - p0) x d1) . dd2 - ((p1 - p0) x d2) . dd1
        num_grad = (
            cr @ (dp1 - dp0)
            + np.cross(p1 - p0, d1) @ dd2
            - np.cross(p1 - p0, d2) @ dd1
        )
        J_delta[fi] = num_grad / (cr_norm * l_bar)
    return deltas, J_delta


xyz_pl_init, _, f_pl_init, Jx0, Jy0, Jz0, *_ = fdm_sensitivities(C, free_pl, supports_pl, xyz0, p_pl, q_pl0)
flat_init, _ = quad_flatness_and_jac(xyz_pl_init, Jx0, Jy0, Jz0)

q_pl = q_pl0.copy()
for step in range(45):
    xyz_pc, _, f_pc, Jxp, Jyp, Jzp, *_ = fdm_sensitivities(C, free_pl, supports_pl, xyz0, p_pl, q_pl)
    deltas, J_del = quad_flatness_and_jac(xyz_pc, Jxp, Jyp, Jzp)

    # combine planarity goal (weight 10.0) with light shape anchor to initial funicular z (weight 0.15)
    r_pl = np.concatenate([10.0 * deltas, 0.15 * (xyz_pc[free_pl, 2] - xyz_pl_init[free_pl, 2])])
    J_pl = np.vstack([10.0 * J_del, 0.15 * Jzp])

    dq_pl = -np.linalg.solve(J_pl.T @ J_pl + 5e-3 * np.eye(len(q_pl)), J_pl.T @ r_pl)
    q_pl = np.clip(q_pl + 0.65 * dq_pl, -45.0, -0.08)

xyz_pl_opt, _, f_pl_opt, Jxf, Jyf, Jzf, *_ = fdm_sensitivities(C, free_pl, supports_pl, xyz0, p_pl, q_pl)
flat_opt, _ = quad_flatness_and_jac(xyz_pl_opt, Jxf, Jyf, Jzf)

in_tol_init = int(np.sum(np.abs(flat_init) <= 0.01))
in_tol_opt = int(np.sum(np.abs(flat_opt) <= 0.01))
print("initial max quad warp   =", round(np.abs(flat_init).max() * 100.0, 2), "%   panels <= 1%:", in_tol_init, "/ 36")
print("planarised max quad warp=", round(np.abs(flat_opt).max() * 100.0, 3), "%   panels <= 1%:", in_tol_opt, "/ 36")
print("all bars stay in compression: max force =", round(f_pl_opt.max(), 3), "kN")

# show initial warped shell on left and planarised compression shell on right
xyz_left = xyz_pl_init.copy()
xyz_right = xyz_pl_opt.copy()
xyz_right[:, 0] += 8.0

f_abs_pl = np.abs(f_pl_opt)
colors_pl = [Color(0.65 - 0.65 * fv / f_abs_pl.max(), 0.65 - 0.65 * fv / f_abs_pl.max(), 0.65 - 0.65 * fv / f_abs_pl.max(), 1.0) for fv in f_abs_pl]

session_pl = Session("fdm_ext04_planar_gridshell")
session_pl.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(xyz_left.tolist()), [GRAY] * len(edges), [2.0] * len(edges))
session_pl.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(xyz_right.tolist()), colors_pl, [3.0] * len(edges))
for net in range(2):
    for k in range(n * n):
        pt = session_pl.objects.points[net * n * n + k]
        pt.pointcolor = BLUE if k in supports_pl else RED
        pt.width = 12.0 if k in supports_pl else 6.0

show(session_pl, "fdm_ext04_planar_gridshell")

initial max quad warp   = 10.93 %   panels <= 1%: 12 / 36
planarised max quad warp= 0.029 %   panels <= 1%: 36 / 36
all bars stay in compression: max force = -0.133 kN


# Observation

**The three line Jacobian $J_x = -D_N^{-1}\,C_N^{T}\,U$ turns forward Force Density equilibrium into an exact differentiable design layer.** Every iterate stays in strict static equilibrium while Gauss Newton updates match target elevations, equalise all 10 bottom chord forces to five decimals, or flatten 36 doubly curved gridshell panels inside architectural glass tolerances.